# Intro: A Minimal Local Eval Harness

This notebook shows the basic learning loop: load a small dataset, run a deterministic mock model, score the outputs, aggregate the results, and write a report.

In [ ]:
from pathlib import Path
import json

from eval_harness.aggregation import summarize_records
from eval_harness.dataset import load_jsonl_cases
from eval_harness.metrics import exact_match, normalized_exact_match, pass_fail_score
from eval_harness.reporting import write_json_report
from eval_harness.runner import run_cases

root = Path.cwd()
cases = load_jsonl_cases(root / "tests" / "evals" / "datasets" / "classification_cases.jsonl")
mock_outputs = json.loads((root / "tests" / "evals" / "mocks" / "mock_model_outputs.json").read_text())

def predict(case):
    return mock_outputs[case.case_id]

def score(case, actual):
    exact = pass_fail_score(exact_match(actual, case.expected))
    normalized = pass_fail_score(normalized_exact_match(actual, case.expected))
    passed = bool(normalized if case.metadata.get("match_type") == "normalized" else exact)
    return {"exact_match": exact, "normalized_exact_match": normalized}, passed

records = run_cases(cases, predict, scorer=score)
summary = summarize_records(records)
write_json_report(root / "outputs" / "notebook_intro_report.json", records, summary)
summary.model_dump()


## What You Will Learn

- How the repo loads a deterministic eval dataset.
- How a local predictor is run and scored.
- How records are aggregated into a summary report.
- Which files matter when you add or debug a simple eval case.

## Prerequisites

- The package is installed with `python3 -m pip install -e ".[dev]".
- The repo passes `make verify` before you start changing behavior.
- You are comfortable running one notebook cell at a time and reading pytest output.

## Key Repo Files
- `tests/evals/datasets/classification_cases.jsonl`
- `tests/evals/mocks/mock_model_outputs.json`
- `src/eval_harness/runner.py`
- `src/eval_harness/metrics.py`

The notebook shows the basic learning loop: load a small dataset, run a deterministic mock model, score the outputs, aggregate the results, and write a report.

In JupyterLab, do this:

1. Open notebooks/01_intro_harness.ipynb.
2. Click the first markdown cell.
3. Replace its contents with the block above.
4. Save with Cmd+S.

Then verify from a terminal:

```
cd $HOME/human-centered-eval-harnesses
make notebooks
```

Then make this a clean commit. You already have other local edits, so do not use git add .. Stage only the notebook:

```
git add notebooks/01_intro_harness.ipynb
git commit -m "docs: add learning goals to intro harness notebook"
git push
```